# Stage 1 — Temperature 1.0, one answer per problem
Evaluate **epoch 5** on **AIME 2024, AIME 2025, AIME 2026, AMC 2023, and MATH-500**.
The `sample1` profile generates exactly **one answer per problem** at **temperature 1.0**
and **top-p 0.7**: **630 responses per checkpoint**, reporting accuracy/pass@1.
Top-p matches the original sampled baseline; the output limit remains **20,480 tokens**.
This means one sampled answer, not one few-shot demonstration in the prompt.

Epoch 5 is selected by default. Enable `INCLUDE_EARLIER_EPOCHS` to evaluate epochs 1–4
afterward. Existing checkpoints are loaded from Drive; this notebook does not train.
Use the **RTX PRO 6000 Blackwell 96GB** runtime and enable the `HF_TOKEN` secret.
Push the new notebook, profile config, and profile-loader changes to the configured
GitHub repository before running setup. Setup clones/updates that repository.

Results are saved under `profiles/sample1/`, separately from greedy and sample8 results.
A baseline run is **not required** to start. The same `sample1` profile is also available
to the baseline CLI when you want a matched comparison. Default Run all performs setup
and checkpoint checks; enable `RUN_STAGE1_EVAL` in the final cell to generate answers.

In [ ]:
REPO_URL = "https://github.com/seungjun-green/post-training-aime.git"
CODE_ROOT = "/content/lg-aime-stage1-eval"
TRAIN_ROOT = "/content/drive/MyDrive/LG-AIME-Stage1"
BASELINE_ROOT = "/content/drive/MyDrive/LG-AIME-English-Eval-compatible"
EVAL_ENV = "/content/lg-eval-env"
CONFIG = "configs/stage1_sft.yaml"
EVAL_PROFILE = "sample1"  # temperature 1.0; one answer on every benchmark

In [ ]:
import os, subprocess, sys, json
from pathlib import Path
from google.colab import drive, userdata
drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
if not Path(CODE_ROOT).exists():
    subprocess.check_call(["git", "clone", "--branch", "main", REPO_URL, CODE_ROOT])
def git(*args):
    return subprocess.check_output(["git", "-C", CODE_ROOT, *args], text=True).strip()
if git("remote", "get-url", "origin") != REPO_URL or git("status", "--porcelain"):
    raise ValueError("Use a clean checkout of the configured repository")
if git("branch", "--show-current") != "main":
    raise ValueError("Use a main checkout")
subprocess.check_call(["git", "-C", CODE_ROOT, "pull", "--ff-only", "origin", "main"])
print("Evaluation code commit:", git("rev-parse", "HEAD"))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "uv==0.11.22"])
subprocess.check_call([sys.executable, "scripts/setup_eval_runtime.py", "--venv", EVAL_ENV], cwd=CODE_ROOT)
sys.path.insert(0, CODE_ROOT)
from common.process import run_logged

## Check saved checkpoints
Keep `TRAIN_ROOT` the same as in the training notebook. SFT evaluation can run before
the baseline. The first run creates the selected profile's protocol/runtime manifests
under `BASELINE_ROOT/profiles/<profile>/full/results/`. Later runs validate the same
sampling, prompts, datasets, scoring and runtime for comparisons within that profile.
To switch from an older notebook, stop evaluation, push the updated code, and run setup.
New profiles start fresh and leave previous 32-sample results intact; old responses are
not imported. Once a run starts, keep its code commit and settings unchanged for resume.

In [ ]:
import yaml
from eval.profiles import profile_root
INCLUDE_EARLIER_EPOCHS = False
cfg = yaml.safe_load((Path(CODE_ROOT) / CONFIG).read_text())
epochs = int(cfg["training"]["num_train_epochs"])
selected_epochs = [epochs]
if INCLUDE_EARLIER_EPOCHS:
    selected_epochs.extend(range(1, epochs))
checkpoint_root = Path(TRAIN_ROOT) / "checkpoints" / cfg["stage"] / cfg["run_name"]
result_root = profile_root(BASELINE_ROOT, EVAL_PROFILE) / "full/results"
run_identities = set()
eval_runs = []
for epoch in selected_epochs:
    checkpoint = checkpoint_root / f"epoch_{epoch}"
    marker = json.loads((checkpoint / "stage1_checkpoint.json").read_text())
    if marker["epoch"] != epoch:
        raise ValueError(f"Checkpoint epoch mismatch: {checkpoint}")
    run_identities.add(marker["run_identity"])
    # Profiles use separate directories for baseline and SFT comparisons.
    run_name = cfg["run_name"] if epoch == epochs else f"{cfg['run_name']}_epoch{epoch}"
    eval_runs.append((epoch, checkpoint, run_name))
if len(run_identities) != 1:
    raise ValueError("Selected epoch checkpoints must belong to the same training run")
suite = json.loads((Path(CODE_ROOT) / "configs/english_eval_suite.json").read_text())
progress_totals = {name: entry["rows"] for name, entry in suite["datasets"].items()}
print("Benchmarks:", ", ".join(progress_totals))
for epoch, checkpoint, run_name in eval_runs:
    print(f"Epoch {epoch}: {checkpoint} -> {result_root / cfg['stage'] / (run_name + '.json')}")

## Full evaluation — epoch 5, then optional earlier checkpoints
Enable `RUN_STAGE1_EVAL` to evaluate epoch 5. Leave `INCLUDE_EARLIER_EPOCHS = False`
above for the main result only. To include the earlier checkpoints, set it to `True`
and rerun the check cell; the evaluation order will be **5, 1, 2, 3, 4**.
A default Run all performs setup and checks only. Every selected epoch uses all five benchmarks, with the same
prompts, sampling, seeds and scoring as the baseline. Epoch 5 remains the Stage 1 result.
Reports, raw generations and manifests are saved under `BASELINE_ROOT/profiles/<profile>/full/results/stage1/`:
`sft_s1k_epoch1.json` through `sft_s1k_epoch4.json`, and `sft_s1k.json` for epoch 5. Completed problems are durably saved during the run
in `*_problems.jsonl`; the standard `*_generations.jsonl` export is written at completion.
Console logs are saved under `TRAIN_ROOT`. Repeating the same commands resumes completed
problems. `configs/eval_profiles.yaml` defines `sample1`: temperature 1.0, top-p 0.7,
one answer per problem, and up to 64 pending problems for 32 GPU response slots. Exact sampled text can
change with batching even though seeds and sampling settings are unchanged.
Progress counts completed problems, which can finish out of order within each benchmark.
No AMC-only runs are launched by this notebook.

In [ ]:
RUN_STAGE1_EVAL = False
if RUN_STAGE1_EVAL:
    eval_python = str(Path(EVAL_ENV) / "bin/python")
    for epoch, checkpoint, run_name in eval_runs:
        command = [eval_python, "-m", "eval.run_batched_eval", "--model", str(checkpoint),
                   "--stage", cfg["stage"], "--run_name", run_name, "--output_root", BASELINE_ROOT,
                   "--profile", EVAL_PROFILE,
                   "--execution_config", "configs/eval_execution.yaml"]
        run_logged(command,
                   cwd=CODE_ROOT, log_path=Path(TRAIN_ROOT) / f"full_eval_epoch{epoch}_{EVAL_PROFILE}_console.log",
                   progress_totals=progress_totals)
else:
    print("Evaluation is off. Enable RUN_STAGE1_EVAL to run the selected checkpoints, starting with epoch 5.")